### ЗАДАЧА: Пакетная обработка переводов между кошельками (exceptions + business rules)

Есть набор кошельков и список входящих переводов.
Нужно безопасно обработать пакет операций: валидные переводы применить к балансам,
ошибочные сохранить в отчёт, не останавливая всю обработку.

НЕОБХОДИМО РЕАЛИЗОВАТЬ:

1. Иерархию кастомных исключений:
   - `TransferError`
   - `TransferFormatError`
   - `AccountNotFoundError`
   - `CurrencyMismatchError`
   - `InsufficientFundsError`
   - `TransferAmountError`.

2. Функцию `parse_transfer(raw)`:
   - формат строки: `transfer_id|from_user|to_user|amount`
   - `amount` должен быть числом и `> 0`
   - при ошибке конвертации использовать `raise ... from ...`.

3. Функцию `apply_transfer(transfer, wallets)`:
   - проверить, что оба пользователя существуют
   - нельзя переводить самому себе
   - валюты кошельков отправителя и получателя должны совпадать
   - у отправителя должно хватать средств
   - при успехе обновить балансы в `wallets`
   - вернуть краткий словарь результата.

4. Функцию `process_batch(rows, wallets)`:
   - для каждой строки вызвать `parse_transfer`, потом `apply_transfer`
   - вернуть `(successes, errors)`
   - ошибки хранить как `(raw, error_type, message)`
   - не прерывать цикл на первой ошибке.

5. Вывести:
   - успешные переводы,
   - ошибки по типам,
   - итоговые балансы,
   - пользователя с максимальным балансом в валюте `USD`.


In [ ]:
wallets = {
    'alice': {'currency': 'USD', 'balance': 1200.0},
    'bob': {'currency': 'USD', 'balance': 450.0},
    'carol': {'currency': 'EUR', 'balance': 900.0},
    'dave': {'currency': 'USD', 'balance': 150.0},
}

rows = [
    'TR-100|alice|bob|200',
    'TR-101|bob|dave|700',
    'TR-102|alice|carol|50',
    'TR-103|eve|bob|30',
    'TR-104|dave|dave|10',
    'TR-105|bob|alice|abc',
    'TR-106|bob|dave|100',
]


class TransferError(Exception):
    pass


class TransferFormatError(TransferError):
    pass


class AccountNotFoundError(TransferError):
    pass


class CurrencyMismatchError(TransferError):
    pass


class InsufficientFundsError(TransferError):
    pass


class TransferAmountError(TransferError):
    pass


def parse_transfer(raw):
    # TODO: распарсить строку и вернуть dict перевода
    parts = raw.split("|")
    if len(parts) != 4:
        raise TransferFormatError("Неверный формат строки")
    transfer_id, from_user, to_user, amount = parts
    
    try:
        amount = int(amount)
    except ValueError as e:
        raise TransferAmountError ("Сумма перевода должна быть числом")
    if amount <=0:
        raise TransferAmountError("Сумма перевода должна быть > 0")
    return {
        "transfer_id": transfer_id,
        "from_user": from_user,
        "to_user": to_user,
        "amount": amount
    }
    # TODO: при ошибке конвертации amount использовать raise ... from ...

def apply_transfer(transfer, wallets):
    from_user = transfer["from_user"]
    to_user = transfer["to_user"]
    amount = transfer["amount"]

    # TODO: проверить существование аккаунтов
    if from_user not in wallets:
        raise AccountNotFoundError(f"Аккаунт отправителя {from_user} не найден")
    if to_user not in wallets:
        raise AccountNotFoundError(f"Аккаунт получателя не найден")
    
    # TODO: запретить перевод самому себе
    if from_user == to_user:
        raise TransferError("Перевод самому себе запрещен")
    
    # TODO: проверить совпадение валют
    from_currency = wallets[from_user]['currency']
    to_currency = wallets[to_user]['currency']
    if from_currency != to_currency:
        raise CurrencyMismatchError("Валюты отправителя и получателя не совпадают")
    
    # TODO: проверить баланс отправителя
    if wallets[from_user]['balance'] < amount:
        raise InsufficientFundsError("Недостаточно средств")
    
    # TODO: обновить балансы и вернуть dict результата
    wallets[from_user]['balance'] -= amount
    wallets[to_user]['balance'] += amount

    return {
        "transfer_id": transfer['transfer_id'],
        "from_user": from_user,
        "to_user": to_user,
        "amount": amount,
        "status": "success"
    }


def process_batch(rows, wallets):
    # TODO: вернуть (successes, errors)
    successes = []
    errors = []
    for raw_transfer in rows:
        try:
            success = parse_transfer(raw_transfer)
            result = apply_transfer(success, wallets)
            successes.append(result)
        except TransferError as e:
            errors.append({
                "raw": raw_transfer,
                "error_type": type(e).__name__,
                "message": e
            })
    return successes, errors


# TODO: вызвать process_batch(rows, wallets)
successes, errors = process_batch(rows, wallets)

# TODO: вывести успешные переводы
print("Успешные переводы: ")
for success in successes:
    print(f"{success["transfer_id"]}:{success["from_user"]} --> {success["to_user"]} ")

# TODO: вывести ошибки по типам
print("Ошибки по типам: ")
error_counts = {}
for error in errors:
    error_type = error['error_type']
    error_counts[error_type] = error_counts.get(error_type, 0) + 1
    print(f"{error["error_type"]}: {error["message"]}")

# TODO: вывести итоговые балансы
print("Итоговые балансы: ")
for user, data in wallets.items():
    print(f"{user}: {data['balance']} {data['currency']}")

# TODO: найти richest_usd_user
richest_usd_user = max((user for user, data in wallets.items() if data['currency'] == 'USD'), key=lambda user: wallets[user]['balance'])
print(f"Пользователь с максимальным балансом USD: {richest_usd_user}")
